# Introduction to population analysis with IBL data

## Load data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from one.api import ONE
from brainbox.io.one import SpikeSortingLoader
from iblatlas.atlas import AllenAtlas

one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    silent=True
)
ba = AllenAtlas()

SPIKESORT_REVISION = "2024-05-06"

eid = '111c1762-7908-47e0-9f40-2f2ee55b6505'
pid = one.eid2pid(eid)[0]

ssl = SpikeSortingLoader(pid=pid, one=one, atlas=ba)

print(f"Probe name: {ssl.pname}")
print(f"Session ID: {ssl.eid}")

spikes, clusters, channels = ssl.load_spike_sorting(revision=SPIKESORT_REVISION)
clusters = ssl.merge_clusters(spikes, clusters, channels)


Load trials data

In [ ]:
trials = one.load_object(eid, 'trials', collection='alf')

## Constructing a population matrix

In [ ]:
cluster_ids = np.asarray(clusters["cluster_id"])
cluster_regions = np.asarray(clusters["acronym"])


Population activity in response to stimulus

In [ ]:
regions_on_probe = np.unique(cluster_regions)

print(f'{len(regions_on_probe)} unique regions sampled on probe')

For population analysis, include only regions with sufficient neurons e.g. n>10

In [ ]:
MIN_NEURONS = 10

REGION_CLUSTERS = {}

for filter_region in regions_on_probe:
    cids_in_region = cluster_ids[cluster_regions == filter_region]

    print(f"{len(cids_in_region)} clusters found in "f"{filter_region}")

    if len(cids_in_region)>MIN_NEURONS:
        REGION_CLUSTERS[filter_region] = cids_in_region

regions_on_probe_pass = np.array(list(REGION_CLUSTERS.keys()))

### Define analysis time windows

For example, 200 ms before and after the stimulus onset

In [ ]:
window_start = -0.2
window_end = 0.2

trial_event_times = np.asarray(trials['stimOn_times'])

n_trials = len(trial_event_times)

Create matrix for a single region

In [ ]:
filter_region = regions_on_probe_pass[0]
cluster_ids = REGION_CLUSTERS[filter_region]
n_neurons = len(cluster_ids)


neuron_spikes = {neuron_id: spikes["times"][spikes["clusters"] == neuron_id] for neuron_id in cluster_ids}

X = np.zeros((n_trials, len(cluster_ids)))
for trial_idx, event_time in enumerate(trials["stimOn_times"]):
    start, end = event_time + window_start, event_time + window_end
    for neuron_idx, neuron_id in enumerate(cluster_ids):
        times = neuron_spikes[neuron_id]
        X[trial_idx, neuron_idx] = np.searchsorted(times, end) - np.searchsorted(times, start)

print(f"Population matrix shape: {X.shape}")


Convert spike counts to firing rates

In [ ]:
window_duration = (window_end - window_start)

X_fr = X / window_duration # Hz for durations in s


In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

im = ax.imshow(
    X_fr,
    aspect="auto",
    interpolation="none",
    cmap="viridis"
)

ax.set_xlabel("Neuron")
ax.set_ylabel("Trial")

fig.colorbar(im,ax=ax,label="Firing rate (Hz)")

plt.show()


## Dimensionality reduction with PCA analysis

Principal component analysis (PCA) can be used to reduce the high-dimensional population activity to a small number of dimensions.

First standardise the neural activity across neurons and then project the population firing rates onto the first 3 principal components that capture the largest sources of variance.

It is important to standardise firing rates across neurons to avoid high firing neurons dominating the population analysis

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(X_fr)

pca = PCA(n_components=3) 

X_pca = pca.fit_transform(X_scaled)


Compute variance explained

In [ ]:
for i, variance in enumerate(pca.explained_variance_ratio_):

    print(f"PC{i + 1}: {variance * 100:.1f}%")


In [ ]:

variance = pca.explained_variance_ratio_ * 100
pcs = np.arange(1, len(variance) + 1)

plt.figure(figsize=(8, 4))

plt.bar(pcs, variance, color='lightgrey')

plt.xlabel("Principal component")
plt.ylabel("Explained variance (%)")
plt.title("Explained variance by principal component")

plt.show()


Comparing between PCs

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(X_pca[:, 0], X_pca[:, 1], s=10, alpha=0.5)

# label by variable eg contrast level

sc = ax.scatter(X_pca[:, 0], X_pca[:, 1],  c=trials["contrastRight"], cmap="coolwarm", s=10, alpha=0.6)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_title("Population activity coloured by signed contrast")
plt.colorbar(sc, ax=ax, label="Signed contrast")
plt.show()


## Decoding task variables from population activity

Example: Using a logistic regression decoder to test whether neural activity can predict the animal’s choice.

Train the decoder on population firing rates and evaluate its performance using 5-fold cross-validation to estimate how much information about choice is contained in the recorded neural activity.

Decode choice side

In [ ]:
y = (trials["choice"] == 1).astype(int)

valid = (trials["choice"] != 0)

X_decode = X_fr[valid]
y_decode = y[valid]


Train a simple logistic regression

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

decoder = make_pipeline(StandardScaler(),LogisticRegression(max_iter=5000, C=1.0))

scores = cross_val_score(decoder, X_decode, y_decode, cv=5)

print(f"Cross-validated accuracy: {scores.mean():.3f} +/- {scores.std():.3f}")